# Deutsches microWakeWord in Google Colab

**Nur Text eingeben; keine Lautschrift nötig.** Das Training läuft in Colab und legt die fertigen `.tflite`- und `.json`-Dateien in Google Drive ab.

1. In Colab eine Laufzeit mit **A100 GPU und hohem RAM** wählen.
2. In der nächsten Zelle `WAKE_WORD` eingeben. `SPOKEN_TEXT` und `SIMILAR_WORDS` sind optional.
3. **Laufzeit → Alle ausführen**. Beim ersten Durchlauf Drive verbinden.
4. Die Hörproben von Thorsten, Pavoque und Ramona anhören. Nur mit `JA` fortfahren, wenn das Wort bei allen drei Stimmen stimmt.
5. Nach dem Training das Modellpaar aus Google Drive herunterladen oder direkt zum Sprachsatelliten kopieren.

Kurze, deutlich unterscheidbare Wörter oder Phrasen eignen sich am besten. Lange Sätze passen nicht zuverlässig in das Trainingsfenster. Weitere Hinweise stehen in den beiden README-Dateien des Repositories.

In [ ]:
# @title 1. Wake word settings (only WAKE_WORD is required)
WAKE_WORD = "Ey Sebastian"  # @param {type:"string"}
SPOKEN_TEXT = ""  # @param {type:"string"}
SIMILAR_WORDS = ""  # @param {type:"string"}

# Optional pronunciation spelling changes only the synthetic speech, not the model name.
import hashlib
import json
import re
import unicodedata

WAKE_WORD = " ".join(WAKE_WORD.split())
SPOKEN_TEXT = " ".join(SPOKEN_TEXT.split()) or WAKE_WORD
assert WAKE_WORD, "Enter a wake word."
assert len(WAKE_WORD) <= 60, "Use a short wake word (at most 60 characters)."
assert len(SPOKEN_TEXT) <= 60, "Use a short spoken text (at most 60 characters)."

def slugify_german(value):
    value = value.casefold().translate(str.maketrans({"ä": "ae", "ö": "oe", "ü": "ue", "ß": "ss"}))
    value = unicodedata.normalize("NFKD", value)
    value = value.encode("ascii", "ignore").decode("ascii")
    return re.sub(r"[^a-z0-9]+", "_", value).strip("_")[:48]

OUTPUT_NAME = slugify_german(WAKE_WORD)
assert OUTPUT_NAME, "The wake word needs letters or digits in its file name."

# Similar words are comma-separated. Exclude phrases containing the exact wake word.
BASE_NEGATIVES = ("Alexa", "Hey Siri", "Okay Google", "Hallo", "Guten Morgen", "Licht an")
user_negatives = [" ".join(part.split()) for part in SIMILAR_WORDS.split(",")]
candidate_negatives = [*BASE_NEGATIVES, *(part for part in user_negatives if part)]
def contains_target(phrase):
    return bool(re.search(r"(?<!\w)" + re.escape(WAKE_WORD) + r"(?!\w)", phrase, re.IGNORECASE))

CONFUSABLE_PHRASES = list(dict.fromkeys(
    phrase for phrase in candidate_negatives if not contains_target(phrase)
))
assert CONFUSABLE_PHRASES, "Provide a distinct negative phrase in SIMILAR_WORDS."

# Three German Piper voices are always used for both preview and training.
VOICE_SAMPLE_COUNTS = {"thorsten": 8000, "pavoque": 6000, "ramona": 6000}
VOICE_SPECS = {
    "thorsten": "thorsten/medium/de_DE-thorsten-medium",
    "pavoque": "pavoque/low/de_DE-pavoque-low",
    "ramona": "ramona/low/de_DE-ramona-low",
}

# A hash prevents previous samples from another word or pronunciation being reused.
RUN_ID = hashlib.sha256(json.dumps(
    [WAKE_WORD, SPOKEN_TEXT, CONFUSABLE_PHRASES, VOICE_SAMPLE_COUNTS],
    ensure_ascii=False, sort_keys=True).encode("utf-8")).hexdigest()[:10]
WORK_DIR = f"/content/wakeword_work/{OUTPUT_NAME}_{RUN_ID}"
DRIVE_FOLDER = f"wakeword_training_de_{OUTPUT_NAME}"
AUTHOR = "Community trainer"
AUTHOR_WEBSITE = ""
SAMPLES_PER_CONFUSABLE = 500
BACKGROUND_TRACK_LIMIT = 500
DOWNLOAD_ROOM_IR = False
PROBABILITY_CUTOFF = 0.85
SLIDING_WINDOW_SIZE = 5
TENSOR_ARENA_SIZE = 50000
TRAINED_LANGUAGES = ["de"]

print(f"Wake word: {WAKE_WORD!r}; spoken input: {SPOKEN_TEXT!r}")
print(f"Similar negatives: {CONFUSABLE_PHRASES}")
print(f"Models: {', '.join(VOICE_SAMPLE_COUNTS)}; Drive folder: {DRIVE_FOLDER}")

In [ ]:
# === Mount Google Drive for the exported model pair ===
from google.colab import drive
import os
drive.mount('/content/drive')
DRIVE_DIR = f'/content/drive/MyDrive/{DRIVE_FOLDER}'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive folder: {DRIVE_DIR}')


In [ ]:
# === Install the pinned Open Home Foundation training framework ===
import importlib
import os
import subprocess
import sys

os.chdir('/content')
DEPS = [
    'audiomentations', 'audio_metadata', 'datasets', 'mmap_ninja',
    'numpy>=2.0', 'pymicro-features', 'pyyaml', 'tensorflow>=2.18',
    'webrtcvad-wheels', 'ai-edge-litert',
    'git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *DEPS], check=True)

MWW_DIR = '/content/ohf-micro-wake-word'
MWW_REVISION = '4665173cd35f1cff9a61e06fc427f124766c488e'
if not os.path.isdir(os.path.join(MWW_DIR, '.git')):
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/OHF-Voice/micro-wake-word.git', MWW_DIR], check=True)
current = subprocess.run(['git', '-C', MWW_DIR, 'rev-parse', 'HEAD'],
                         check=True, capture_output=True, text=True).stdout.strip()
if current != MWW_REVISION:
    subprocess.run(['git', '-C', MWW_DIR, 'fetch', '--depth', '1',
                    'origin', MWW_REVISION], check=True)
    subprocess.run(['git', '-C', MWW_DIR, 'checkout', '--detach', 'FETCH_HEAD'], check=True)
assert subprocess.run(['git', '-C', MWW_DIR, 'rev-parse', 'HEAD'],
                      check=True, capture_output=True, text=True).stdout.strip() == MWW_REVISION
if MWW_DIR not in sys.path:
    sys.path.insert(0, MWW_DIR)
importlib.invalidate_caches()
from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration
print(f'Open Home Foundation micro-wake-word ready at {MWW_REVISION[:12]}')


In [ ]:
# === Isolate temporary Colab data for this wake word and pronunciation ===
import os
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)
print(f'Temporary training files: {WORK_DIR}')


In [ ]:
# === Download German Piper voices and check pronunciation ===
import json, subprocess, sys
from pathlib import Path
from huggingface_hub import hf_hub_download
from IPython.display import Audio, display

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'piper-sample-generator==3.2.0'], check=True)
import importlib.metadata
generator_version = importlib.metadata.version('piper-sample-generator')
assert generator_version == '3.2.0', f'Unexpected generator version: {generator_version}'
module_path = Path(importlib.metadata.distribution('piper-sample-generator').locate_file(
    'piper_sample_generator/__main__.py'))
generator_source = module_path.read_text(encoding='utf-8')
legacy_import = ('try:\n'
                 '    from piper_train.vits import commons\n'
                 'except ImportError:\n'
                 '    from piper_train.vits import commons')
legacy_marker = 'commons = None  # The legacy PyTorch generator is unused for ONNX voices.'
if legacy_import in generator_source:
    preserved_import = '\n'.join(
        f'# [REMOVED: ONNX does not need piper_train] {line}'
        for line in legacy_import.splitlines())
    generator_source = generator_source.replace(
        legacy_import, preserved_import + '\n' + legacy_marker, 1)
    module_path.write_text(generator_source, encoding='utf-8')
    print(f'Patched the ONNX import path in {module_path}')
elif legacy_marker not in generator_source:
    raise RuntimeError('The sample generator import layout changed; inspect it before patching.')
sys.modules.pop('piper_sample_generator.__main__', None)
from piper_sample_generator.__main__ import generate_samples_onnx

VOICE_PATHS = {}
for label, relative in VOICE_SPECS.items():
    filename = f'de/de_DE/{relative}.onnx'
    voice = hf_hub_download('rhasspy/piper-voices', filename=filename,
                            local_dir='/content/models')
    voice_config = hf_hub_download('rhasspy/piper-voices',
                                    filename=f'{filename}.json',
                                    local_dir='/content/models')
    assert Path(voice).stat().st_size > 1_000_000, f'Incomplete voice: {voice}'
    assert Path(voice_config).is_file()
    with open(voice_config, encoding='utf-8') as config_file:
        voice_info = json.load(config_file)
    VOICE_PATHS[label] = voice
    print(f'{label}: {voice_info["num_speakers"]} speaker(s), '
          f'{voice_info["audio"]["sample_rate"]} Hz')

preview_dir = Path(WORK_DIR) / 'pronunciation_preview'
preview_dir.mkdir(exist_ok=True)
for voice_label in VOICE_SAMPLE_COUNTS:
    preview_path = preview_dir / voice_label
    preview_path.mkdir(parents=True, exist_ok=True)
    generate_samples_onnx(SPOKEN_TEXT, preview_path,
                          model=VOICE_PATHS[voice_label], max_samples=1,
                          phoneme_input=False)
    print(f'Preview: {WAKE_WORD} / {voice_label} / spoken text={SPOKEN_TEXT!r}')
    display(Audio(str(preview_path / '0.wav')))
answer = input('Do ALL three voices pronounce your wake word correctly? Type JA: ')
assert answer.strip().upper() == 'JA', 'Stopped before generating the full dataset.'
print('German pronunciation approved')


In [ ]:
# === Generate positive German speech samples ===
import shutil, subprocess, sys
from pathlib import Path
positive_dir = Path(WORK_DIR) / 'generated_samples'
positive_dir.mkdir(exist_ok=True)
for label, count in VOICE_SAMPLE_COUNTS.items():
    existing = len(list(positive_dir.glob(f'{label}_*.wav')))
    if existing == count:
        print(f'{label}: {count} positive WAVs already present')
        continue
    temporary_dir = Path(WORK_DIR) / f'_positive_{label}'
    shutil.rmtree(temporary_dir, ignore_errors=True)
    print(f'Generating {count} German positives using {label}...')
    cmd = [sys.executable, '-m', 'piper_sample_generator', SPOKEN_TEXT,
           '--model', VOICE_PATHS[label],
           '--max-samples', str(count), '--output-dir', str(temporary_dir),
           '--length-scales', '0.85', '1.0', '1.15',
           '--noise-scales', '0.667', '0.75', '--noise-scale-ws', '0.8']
    subprocess.run(cmd, check=True)
    generated = sorted(temporary_dir.glob('*.wav'))
    assert len(generated) == count, f'{label}: got {len(generated)} instead of {count}'
    for wav in generated:
        wav.replace(positive_dir / f'{label}_{wav.name}')
    shutil.rmtree(temporary_dir)
total = len(list(positive_dir.glob('*.wav')))
assert total == sum(VOICE_SAMPLE_COUNTS.values()), f'Unexpected positive WAV count: {total}'
print(f'Total German positive samples: {total}')


In [ ]:
# === Generate similar phrases that should not activate the model ===
import shutil, subprocess, sys
from pathlib import Path
negative_dir = Path(WORK_DIR) / 'confusable_negatives'
negative_dir.mkdir(exist_ok=True)
active_voices = list(VOICE_SAMPLE_COUNTS)
for index, phrase in enumerate(CONFUSABLE_PHRASES):
    prefix = f'{index:02d}_{slugify_german(phrase)}'
    existing = len(list(negative_dir.glob(f'{prefix}_*.wav')))
    if existing == SAMPLES_PER_CONFUSABLE:
        print(f'{phrase!r}: already generated')
        continue
    temporary_dir = Path(WORK_DIR) / f'_negative_{prefix}'
    shutil.rmtree(temporary_dir, ignore_errors=True)
    voice = active_voices[index % len(active_voices)]
    cmd = [sys.executable, '-m', 'piper_sample_generator', phrase,
           '--model', VOICE_PATHS[voice], '--max-samples',
           str(SAMPLES_PER_CONFUSABLE), '--output-dir', str(temporary_dir),
           '--length-scales', '0.85', '1.0', '1.15',
           '--noise-scales', '0.667', '0.75', '--noise-scale-ws', '0.8']
    print(f'Generating hard negative {phrase!r} with {voice}...')
    subprocess.run(cmd, check=True)
    generated = sorted(temporary_dir.glob('*.wav'))
    assert len(generated) == SAMPLES_PER_CONFUSABLE
    for wav in generated:
        wav.replace(negative_dir / f'{prefix}_{wav.name}')
    shutil.rmtree(temporary_dir)
print(f'Total hard negatives: {len(list(negative_dir.glob("*.wav")))}')


In [ ]:


# MIT room impulse responses for reverb augmentation

# Background noise corpora (FMA + AudioSet 16 kHz subsets)

# === Download published negative features and background audio ===
import random, subprocess, zipfile
from pathlib import Path
from huggingface_hub import hf_hub_download

negative_root = Path('negative_datasets')
negative_root.mkdir(exist_ok=True)
for name in ('speech', 'dinner_party', 'no_speech', 'dinner_party_eval'):
    marker = negative_root / f'.{name}_complete'
    if marker.exists():
        print(f'{name}: already extracted')
        continue
    print(f'Downloading and extracting {name}.zip (several GB for some sets)...')
    archive = hf_hub_download('kahrendt/microwakeword', filename=f'{name}.zip',
                              repo_type='dataset', local_dir=str(negative_root))
    with zipfile.ZipFile(archive) as zf:
        zf.extractall(negative_root)
    assert (negative_root / name).is_dir(), f'Archive did not contain {name}/'
    Path(archive).unlink(missing_ok=True)
    marker.write_text('extracted\n')

# Convert a bounded subset of the published FMA archive to 16 kHz mono WAV.
subprocess.run(['apt-get', '-qq', 'install', '-y', 'ffmpeg'], check=True)
music_root = Path('fma_16k')
music_root.mkdir(exist_ok=True)
if len(list(music_root.glob('*.wav'))) < BACKGROUND_TRACK_LIMIT:
    archive = hf_hub_download('mchl914/fma_xsmall', filename='fma_xs.zip',
                              repo_type='dataset', local_dir='fma')
    with zipfile.ZipFile(archive) as zf:
        zf.extractall('fma')
    tracks = sorted(Path('fma').rglob('*.mp3'))
    assert tracks, 'The FMA archive contained no MP3 tracks.'
    chosen = random.Random(42).sample(tracks, min(BACKGROUND_TRACK_LIMIT, len(tracks)))
    for index, track in enumerate(chosen):
        output = music_root / f'{index:04d}.wav'
        if output.is_file() and output.stat().st_size > 44:
            continue
        subprocess.run(['ffmpeg', '-nostdin', '-hide_banner', '-loglevel', 'error',
                        '-y', '-i', str(track), '-t', '30', '-ar', '16000', '-ac',
                        '1', '-c:a', 'pcm_s16le', str(output)], check=True)
    Path(archive).unlink(missing_ok=True)
    print(f'Converted {len(list(music_root.glob("*.wav")))} background clips')

# OpenSLR room impulse responses are optional due to the 1.3 GB download.
ROOM_IMPULSE_DIRS = []
if DOWNLOAD_ROOM_IR:
    room_root = Path('mit_rirs')
    room_root.mkdir(exist_ok=True)
    if not any(room_root.rglob('*.wav')):
        room_archive = room_root / 'rirs_noises.zip'
        subprocess.run(['wget', '-q', '-c', '-O', str(room_archive),
                        'https://www.openslr.org/resources/28/rirs_noises.zip'], check=True)
        with zipfile.ZipFile(room_archive) as zf:
            for item in zf.infolist():
                if item.filename.lower().endswith('.wav') and 'rir' in item.filename.lower():
                    zf.extract(item, room_root)
        room_archive.unlink(missing_ok=True)
    assert any(room_root.rglob('*.wav')), 'No room impulse WAVs were extracted.'
    ROOM_IMPULSE_DIRS = [str(room_root)]
print('Negative features and background audio are ready')


In [ ]:
# === Augmentation + feature extraction ===
from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration
import os, shutil, traceback
from mmap_ninja.ragged import RaggedMmap

clips = Clips(
    input_directory='generated_samples',
    file_pattern='*.wav',
    max_clip_duration_s=None,
    remove_silence=True,
    random_split_seed=42,
    split_count=0.1,
)
augmenter = Augmentation(
    augmentation_duration_s=3.2,
    augmentation_probabilities={
        'SevenBandParametricEQ': 0.15, 'TanhDistortion': 0.10,
        'PitchShift': 0.15, 'BandStopFilter': 0.10,
        'AddColorNoise': 0.20, 'AddBackgroundNoise': 0.85,
        'Gain': 1.00, 'GainTransition': 0.25, 'RIR': 0.60,
    },
    impulse_paths=ROOM_IMPULSE_DIRS,
    background_paths=['fma_16k'],
    background_min_snr_db=-5, background_max_snr_db=20,
    min_jitter_s=0.10, max_jitter_s=0.50,
)

os.makedirs('generated_augmented_features', exist_ok=True)
SPLIT_CONFIG = {
    'training':   {'split_name': 'train',      'repetition': 3, 'slide_frames': 10},
    'validation': {'split_name': 'validation', 'repetition': 1, 'slide_frames': 10},
    'testing':    {'split_name': 'test',       'repetition': 1, 'slide_frames': 1 },
}

for split, cfg in SPLIT_CONFIG.items():
    out = f'generated_augmented_features/{split}'
    mmap = f'{out}/wakeword_mmap'
    if os.path.exists(mmap) and list(os.scandir(mmap)):
        print(f'{split}: cached, skipping')
        continue
    if os.path.exists(mmap):
        shutil.rmtree(mmap)
    os.makedirs(out, exist_ok=True)
    print(f'Generating {split} (rep={cfg["repetition"]}, slide={cfg["slide_frames"]})...')
    try:
        sg = SpectrogramGeneration(clips=clips, augmenter=augmenter,
                                    slide_frames=cfg['slide_frames'], step_ms=10)
        RaggedMmap.from_generator(
            out_dir=mmap, batch_size=200, verbose=True,
            sample_generator=sg.spectrogram_generator(
                split=cfg['split_name'], repeat=cfg['repetition']),
        )
    except Exception:
        traceback.print_exc()
        if os.path.exists(mmap): shutil.rmtree(mmap)
        raise
print('Positive features ready')

# Confusable features (only if confusable_negatives/ exists)
if os.path.exists('confusable_negatives') and os.listdir('confusable_negatives'):
    print('Generating confusable features...')
    confusable_clips = Clips(
        input_directory='confusable_negatives', file_pattern='*.wav',
        max_clip_duration_s=None, remove_silence=True,
        random_split_seed=42, split_count=0.1,
    )
    os.makedirs('confusable_features', exist_ok=True)
    for split, cfg in SPLIT_CONFIG.items():
        out = f'confusable_features/{split}'
        mmap = f'{out}/wakeword_mmap'
        if os.path.exists(mmap) and list(os.scandir(mmap)):
            continue
        if os.path.exists(mmap): shutil.rmtree(mmap)
        os.makedirs(out, exist_ok=True)
        sg = SpectrogramGeneration(clips=confusable_clips, augmenter=augmenter,
                                    slide_frames=cfg['slide_frames'], step_ms=10)
        RaggedMmap.from_generator(
            out_dir=mmap, batch_size=200, verbose=True,
            sample_generator=sg.spectrogram_generator(
                split=cfg['split_name'], repeat=cfg['repetition']),
        )
    print('Confusable features ready')

# Real recording features (only if real_recordings/ exists)
if os.path.exists('real_recordings') and os.listdir('real_recordings'):
    print('Generating real-recording features...')
    real_clips = Clips(
        input_directory='real_recordings', file_pattern='*.wav',
        max_clip_duration_s=None, remove_silence=True,
        random_split_seed=42, split_count=0.1,
    )
    os.makedirs('real_recording_features', exist_ok=True)
    for split, cfg in SPLIT_CONFIG.items():
        out = f'real_recording_features/{split}'
        mmap = f'{out}/wakeword_mmap'
        if os.path.exists(mmap) and list(os.scandir(mmap)):
            continue
        if os.path.exists(mmap): shutil.rmtree(mmap)
        os.makedirs(out, exist_ok=True)
        sg = SpectrogramGeneration(clips=real_clips, augmenter=augmenter,
                                    slide_frames=cfg['slide_frames'], step_ms=10)
        RaggedMmap.from_generator(
            out_dir=mmap, batch_size=200, verbose=True,
            sample_generator=sg.spectrogram_generator(
                split=cfg['split_name'], repeat=cfg['repetition']),
        )
    print('Real-recording features ready')


In [ ]:
# === Match the training window to the generated word duration ===
import math
import wave
from pathlib import Path

clips_for_duration = sorted(Path('generated_samples').glob('*.wav'))
assert clips_for_duration, 'No generated samples found.'
durations = []
for file in clips_for_duration[::max(1, len(clips_for_duration) // 400)]:
    with wave.open(str(file), 'rb') as wav:
        durations.append(wav.getnframes() / wav.getframerate())
durations.sort()
duration_p95 = durations[min(len(durations) - 1, int(len(durations) * 0.95))]
assert duration_p95 < 3.0, (
    f'The word needs {duration_p95:.2f} s; use a shorter wake word or faster spoken text.')
CLIP_DURATION_MS = max(1500, min(3000, math.ceil((duration_p95 + 0.25) * 10) * 100))
print(f'95th percentile speech: {duration_p95:.2f} s; training window: {CLIP_DURATION_MS} ms')

In [ ]:
# === Training config YAML ===
import yaml, os
from pathlib import Path

SKIP_CONFUSABLES = not Path('confusable_features/training/wakeword_mmap').exists()
SKIP_REAL = not Path('real_recording_features/training/wakeword_mmap').exists()

config = {
    'window_step_ms': 10,
    'train_dir': f'trained_models/{OUTPUT_NAME}',
    'features': [
        dict(features_dir='generated_augmented_features', sampling_weight=8.0,
             penalty_weight=2.0, truth=True, truncation_strategy='truncate_start',
             type='mmap'),
        dict(features_dir='negative_datasets/speech', sampling_weight=10.0,
             penalty_weight=2.5, truth=False, truncation_strategy='random', type='mmap'),
        dict(features_dir='negative_datasets/dinner_party', sampling_weight=15.0,
             penalty_weight=3.0, truth=False, truncation_strategy='random', type='mmap'),
        dict(features_dir='negative_datasets/no_speech', sampling_weight=5.0,
             penalty_weight=1.0, truth=False, truncation_strategy='random', type='mmap'),
        dict(features_dir='negative_datasets/dinner_party_eval', sampling_weight=0.0,
             penalty_weight=1.0, truth=False, truncation_strategy='split', type='mmap'),
    ],
    'training_steps': [25000, 20000],
    'positive_class_weight': [2, 2],
    'negative_class_weight': [40, 50],
    'learning_rates': [0.001, 0.0001],
    'batch_size': 256,
    'time_mask_max_size': [5, 5], 'time_mask_count': [1, 1],
    'freq_mask_max_size': [3, 3], 'freq_mask_count': [1, 1],
    'eval_step_interval': 500,
    'clip_duration_ms': CLIP_DURATION_MS,
    'target_minimization': 0.4,
    'minimization_metric': 'ambient_false_positives_per_hour',
    'maximization_metric': 'average_viable_recall',
}

if not SKIP_CONFUSABLES:
    config['features'].append(dict(features_dir='confusable_features', sampling_weight=8.0,
                                    penalty_weight=5.0, truth=False,
                                    truncation_strategy='random', type='mmap'))
if not SKIP_REAL:
    config['features'].append(dict(features_dir='real_recording_features', sampling_weight=8.0,
                                    penalty_weight=2.0, truth=True,
                                    truncation_strategy='truncate_start', type='mmap'))

os.makedirs(f'trained_models/{OUTPUT_NAME}', exist_ok=True)
with open('training_parameters.yaml', 'w') as f:
    yaml.dump(config, f)
print('training_parameters.yaml ready')
print(f'  Feature sets: {len(config["features"])}, total steps: {sum(config["training_steps"])}')


In [ ]:
# === Train the model ===
# subprocess + PYTHONPATH so it can find microwakeword (sys.path doesn't propagate)
import os, sys, subprocess, shutil
shutil.rmtree(f'trained_models/{OUTPUT_NAME}', ignore_errors=True)

env = os.environ.copy()
env['PYTHONPATH'] = MWW_DIR + ':' + env.get('PYTHONPATH', '')
env['XLA_FLAGS'] = '--xla_gpu_autotune_level=0'

cmd = [
    sys.executable, '-m', 'microwakeword.model_train_eval',
    '--training_config', 'training_parameters.yaml',
    '--train', '1', '--restore_checkpoint', '0',
    '--test_tflite_streaming_quantized', '1',
    '--use_weights', 'best_weights',
    'mixednet',
    '--pointwise_filters', '64,64,64,64',
    '--repeat_in_block', '1, 1, 1, 1',
    '--mixconv_kernel_sizes', '[5], [7,11], [9,15], [23]',
    '--residual_connection', '0,0,0,0',
    '--first_conv_filters', '32',
    '--first_conv_kernel_size', '5',
    '--stride', '3',
]
print('Running:', ' '.join(cmd)); print()
proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
proc.wait()
print(); print('Exit code:', proc.returncode)
assert proc.returncode == 0, 'training failed - see output above'


In [ ]:
# === Export + push to Drive ===
import os, json, shutil, datetime

tflite_src = f'trained_models/{OUTPUT_NAME}/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite'
assert os.path.exists(tflite_src), f'No model at {tflite_src}'

OUT_TFLITE = f'{OUTPUT_NAME}.tflite'
OUT_JSON = f'{OUTPUT_NAME}.json'
shutil.copy2(tflite_src, OUT_TFLITE)
print(f'wrote {OUT_TFLITE} ({os.path.getsize(OUT_TFLITE)/1024:.1f} KB)')

manifest = {
    'type': 'micro',
    'wake_word': WAKE_WORD,
    'author': AUTHOR,
    'website': AUTHOR_WEBSITE,
    'model': OUT_TFLITE,
    'trained_languages': TRAINED_LANGUAGES,
    'version': 2,
    'micro': {
        'probability_cutoff': PROBABILITY_CUTOFF,
        'feature_step_size': 10,
        'sliding_window_size': SLIDING_WINDOW_SIZE,
        'tensor_arena_size': TENSOR_ARENA_SIZE,
        'minimum_esphome_version': '2024.7.0',
    }
}
with open(OUT_JSON, 'w') as f:
    json.dump(manifest, f, indent=2)
print(f'wrote {OUT_JSON}')
print(json.dumps(manifest, indent=2))

for fn in (OUT_TFLITE, OUT_JSON):
    shutil.copy2(fn, f'{DRIVE_DIR}/{fn}')
    print(f'pushed {fn} -> {DRIVE_DIR}')

ts = datetime.datetime.now(datetime.timezone.utc).isoformat()
with open(f'{DRIVE_DIR}/_run_finished.txt', 'w') as f:
    f.write(f'Training run finished at {ts}\n')
print()
print(f'DONE. Find your model at /content/drive/MyDrive/{DRIVE_FOLDER}/{OUTPUT_NAME}.tflite')


## Fertiges Modell

Die beiden Dateien `<name>.tflite` und `<name>.json` liegen im angezeigten Google-Drive-Ordner. Kopiere sie zusammen an den Ort für eigene microWakeWord-Modelle deines ESPHome- oder Linux-Voice-Assistant-Satelliten. Der Dateiname in der JSON-Datei muss zur `.tflite`-Datei passen.

Für ESPHome z. B. beide Dateien nach `/config/esphome/wakewords/<name>/` kopieren und in der Gerätekonfiguration einbinden:

```yaml
micro_wake_word:
  models:
    - model: wakewords/<name>/<name>.json
  on_wake_word_detected:
    - voice_assistant.start:
```

Auf dem Gerät im vorgesehenen Raum testen. Ausspracheschwankungen, Fernfeld, Fernseher und andere Personen sind mit synthetischen Stimmen allein nicht vollständig abgedeckt. Bei Fehlalarmen `SIMILAR_WORDS` gezielt ergänzen und neu trainieren. Details: `README.de.md` und `README.md`.